# Imports

In [1]:
import cda2
import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
import json

from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Connect to Spark

In [2]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [3]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [4]:
api.start_spark(n_executors=400, config=config)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


23/10/31 08:13:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
23/10/31 08:13:40 WARN DomainSocketFactory: The short-circuit local reads feature cannot be used because libhadoop cannot be loaded.
23/10/31 08:13:45 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!


Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string.

In [5]:
@F.udf("string")
def to_date(ts):
    return datetime.datetime.utcfromtimestamp(ts / 1000).strftime("%Y%m%d")

In [59]:
year0 = "2019"
year1 = str(int(year0) + 1)

In [60]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}

In [66]:
df_fixes_raw = (
    api.dataframe("ArincFix", **dates, metadata=True)
    .withColumn("uniq_fix_name", F.concat("identification.name", F.lit("("), "identification.icao_region", F.lit(")")))
    .select(
        "uniq_fix_name",
        F.col("identification.name").alias("fix_name"),
        F.col("arinc_record_info.customer_area_code").alias("area_code"),
        F.col("identification.icao_region").alias("icao_region"),
        "latitude",
        "longitude",
        F.col("navaid_info.dme_latitude").alias("dme_latitude"),
        F.col("navaid_info.dme_longitude").alias("dme_longitude"),
        F.col("magnetic_variation.modeled").alias("magnetic_variation"),
        F.col("metadata.effective_end_date").alias("end_date"),
    )
#    .filter(F.col("latitude") == 0)
#    .filter(F.col("longitude") == 0)
#    .filter(F.col("dme_latitude") == 0)
#    .filter(F.col("dme_longitude") == 0)
    .orderBy("uniq_fix_name")
)

#df_fixes_raw.show()

Multiple versions found: 3.1.12.2, 3.1.22


In [67]:
#df_fixes_raw.count()

In [68]:
#df_fixes_raw.show()

In [69]:
window = Window.partitionBy("uniq_fix_name").orderBy(col("end_date").desc())

df_fixes = (df_fixes_raw
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
)
 
#df_fixes.show()

In [70]:
df_fixes.count()

221316

In [71]:
#df_fixes.show()

In [72]:
(
    df_fixes
    .write.option("header", True)
    .csv("CRAFT/" + year0 + "/fixes", compression="None", mode="overwrite")
)

23/10/31 09:24:41 ERROR TransportClient: Failed to send RPC RPC 4836243779209773416 to /192.168.164.139:37356: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source)
23/10/31 09:24:41 ERROR TransportClient: Failed to send RPC RPC 5131855409433447696 to /192.168.164.128:45590: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source)
23/10/31 09:24:41 ERROR TransportClient: Failed to send RPC RPC 8252440918488986265 to /192.168.164.196:40496: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source)
23/10/31 09:24:41 ERROR TransportClient: Failed to send RPC RPC 536724989926771